# Lab: Ensemble Learning — Bagging, Boosting, and Stacking

**Based on:** [ENSEMBLMethods.ipynb](https://github.com/acesillinois/cpsc-ese-practicum/blob/main/ENSEMBLMethods.ipynb)  
**Time:** ~2 hours | **Total:** 100 points | **Deliverable:** completed Colab notebook

### Guiding question
**When does combining multiple learners improve prediction, and what are the tradeoffs?**

### Learning outcomes
1. Explain how bagging, random forests, boosting, and stacking differ.
2. Compare regression models using baseline, validation, and test errors.
3. Investigate how model complexity influences training and validation error.
4. Use early stopping appropriately without tuning on the test set.
5. Interpret experimental findings and relate them to biological prediction.

**Instructions:** Run cells in order, complete all `TODO` assignments, and answer the questions in Markdown. Keep the final test set untouched until Section 6. Set random seeds to make experiments reproducible.


## Part 1 — Explore the data and establish a baseline (10 points)

The UCI Abalone dataset includes physical measurements and the number of shell rings. We model **Rings**, a proxy for age, not exact chronological age.

**Q1 (3 pts):** What is the response variable? Name three predictors and explain why the relationship might be nonlinear.  
**Q2 (3 pts):** Report sample size, missing values, and the distribution of the response.  
**Q3 (4 pts):** Explain what the baseline model predicts and why a sophisticated model should outperform it.


In [ ]:
# Optional in Colab, if xgboost is absent:
# %pip -q install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

SEED = 42
rng = np.random.default_rng(SEED)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def scores(model, X, y):
    pred = model.predict(X)
    return {'RMSE': rmse(y, pred), 'R2': r2_score(y, pred)}


In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data'
columns = ['Sex','Length','Diameter','Height','Whole weight',
           'Shucked weight','Viscera weight','Shell weight','Rings']
raw = pd.read_csv(url, names=columns)
display(raw.head())
print('Shape:', raw.shape)
print('Missing:', raw.isna().sum().to_dict())
display(raw['Rings'].describe())
raw['Rings'].hist(bins=25)
plt.xlabel('Shell rings'); plt.ylabel('Frequency'); plt.title('Response distribution')
plt.show()


**Data partition:** 70% development and 30% held-out test. Split development again into training (80% of development) and validation (20% of development). The test set is reserved for one final comparison. We encode `Sex` consistently across all partitions; it is categorical, not a continuous physical measurement.

In [ ]:
X = pd.get_dummies(raw.drop(columns='Rings'), columns=['Sex'], dtype=int)
y = raw['Rings']
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.20, random_state=SEED)
print('Training:', len(X_train), 'Validation:', len(X_val), 'Test:', len(X_test))

baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
print('Baseline validation:', scores(baseline, X_val, y_val))


### Student responses — Part 1

**Q1:** _Write here._  
**Q2:** _Write here._  
**Q3:** _Write here._


## Part 2 — Random Forest: bagging and feature randomness (20 points)

A Random Forest averages predictions from many trees trained using bootstrap samples and randomized feature selection. Plain bagging does not require random feature subsets.

**Tasks:** (a) Compare one decision tree with a 100-tree Random Forest (5 pts). (b) Investigate `n_estimators` = 10, 50, 100, 200 (5 pts). (c) Investigate `max_depth` = 2, 4, 8, None (5 pts). (d) Explain why the train–validation gap changes and whether more trees always help (5 pts).


In [ ]:
tree = DecisionTreeRegressor(random_state=SEED).fit(X_train, y_train)
forest = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1).fit(X_train, y_train)
print('Decision tree:', scores(tree, X_val, y_val))
print('Random Forest:', scores(forest, X_val, y_val))

# TODO: loop over tree counts and store training/validation RMSE in a DataFrame.
tree_counts = [10, 50, 100, 200]
rf_count_rows = []
for count in tree_counts:
    # TODO: fit RandomForestRegressor(n_estimators=count, ...)
    # TODO: append {'n_estimators': count, 'train_RMSE': ..., 'val_RMSE': ...}
    pass
# display(pd.DataFrame(rf_count_rows))


In [ ]:
# TODO: study maximum depth, store both training and validation RMSE, and plot them.
depths = [2, 4, 8, None]
rf_depth_rows = []
for depth in depths:
    pass
# display(pd.DataFrame(rf_depth_rows))
# Plot the results with labeled axes and a legend.


### Student responses — Part 2

**(a) Single tree vs forest:** _Write here._  
**(b) Tree count and performance:** _Write here._  
**(c) Depth and performance:** _Write here._  
**(d) Train–validation gap:** _Write here._


## Part 3 — Boosting: learning rate and number of rounds (20 points)

Unlike bagging, boosting builds trees sequentially to reduce the current objective. Explore the interaction between learning rate and the number of boosting rounds.

**Tasks:** (a) Train XGBoost at learning rates 0.03, 0.1, 0.3 and rounds 20, 50, 100, 200 (10 pts). (b) Display a validation RMSE table and plot RMSE versus boosting rounds (5 pts). (c) Explain how lowering the learning rate changes the number of rounds needed, and why more boosting can eventually harm validation performance (5 pts).


In [ ]:
learning_rates = [0.03, 0.1, 0.3]
rounds = [20, 50, 100, 200]
xgb_rows = []
for lr in learning_rates:
    for n_rounds in rounds:
        # TODO: fit XGBRegressor(n_estimators=n_rounds, learning_rate=lr,
        # max_depth=3, objective='reg:squarederror', random_state=SEED,
        # n_jobs=2, tree_method='hist') on X_train/y_train.
        # TODO: record train/validation RMSE for each combination.
        pass
# xgb_results = pd.DataFrame(xgb_rows)
# display(xgb_results)
# TODO: make a line plot of validation RMSE vs rounds, one curve per learning rate.


### Student responses — Part 3

**(a–b) Observed results:** _Include the table and plot._  
**(c) Learning-rate/rounds tradeoff:** _Write here._


## Part 4 — Early stopping (15 points)

Early stopping uses **validation performance**, not test error, to select a useful number of boosting rounds. It is distinct from structural decision-tree pruning, although both can limit complexity.

**Tasks:** (a) Train with a large maximum number of rounds and `early_stopping_rounds=15` (5 pts). (b) Plot training and validation RMSE by iteration (5 pts). (c) Record `best_iteration`, explain why the minimum validation RMSE and the final trained iteration can differ, and discuss how stopping helps (5 pts).


In [ ]:
early_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.1,
    objective='reg:squarederror', random_state=SEED,
    n_jobs=2, tree_method='hist', early_stopping_rounds=15,
    eval_metric='rmse'
)
early_model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
history = early_model.evals_result()
print('Best iteration (zero-indexed):', early_model.best_iteration)
print('Validation RMSE at selected iteration:', scores(early_model, X_val, y_val)['RMSE'])

# TODO: plot history['validation_0']['rmse'] and history['validation_1']['rmse']
# vs iteration. Add a vertical line at early_model.best_iteration.


### Student responses — Part 4

**(a–b) Best iteration and plot:** _Insert here._  
**(c) Why validation monitoring works:** _Write here._


## Part 5 — Stacking: complementary learners (20 points)

Stacking fits a meta-model on predictions from several base models. `StackingRegressor` internally creates out-of-fold predictions from training data for the meta-learner; the external validation partition remains available for comparison. **Important:** SVR depends on feature scale, so it is wrapped in a scaler pipeline.

**Tasks:** (a) Fit Random Forest, scaled SVR, and Ridge as individual models (5 pts). (b) Combine RF and scaled SVR using Ridge as meta-learner (5 pts). (c) Compare validation errors for all models (5 pts). (d) Explain whether stacking improves performance and why model diversity matters (5 pts).


In [ ]:
rf_base = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=SEED, n_jobs=-1)
svr_base = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=10, epsilon=0.2))
ridge_base = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

# TODO: fit each model on X_train/y_train and record its validation RMSE.

stack = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=100, max_depth=8, random_state=SEED, n_jobs=-1)),
        ('svr', make_pipeline(StandardScaler(), SVR(kernel='rbf', C=10, epsilon=0.2)))
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)
# TODO: fit stack and compare validation performance with individual models.


### Student responses — Part 5

**(a–c) Model comparison:** _Include table._  
**(d) Interpretation and complementary models:** _Write here._


## Part 6 — Final evaluation, interpretation, and transfer (15 points)

**Select models using validation results only.** Once the configurations are fixed, you may refit on all development data (`X_dev`, `y_dev`) if the chosen model can be trained without validation-based stopping. For early stopping, retain an independent validation set for stopping or use a predetermined iteration count selected during development. Do **not** select hyperparameters using test-set results.

**Tasks:** (a) Evaluate the mean baseline and your selected forest, boosting model, and stacking model on the same held-out test set (5 pts). (b) Produce an observed-versus-predicted scatterplot and residual plot for your best model (5 pts). (c) Interpret whether the ensemble advantage is substantial, and propose an agroecological application plus one limitation of random splitting for clustered field/temporal data (5 pts).


In [ ]:
# Example final refit; replace hyperparameters based ONLY on previous validation results.
final_models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=8,
                                           random_state=SEED, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1,
                                max_depth=3, objective='reg:squarederror',
                                random_state=SEED, n_jobs=2, tree_method='hist'),
    'Stacking': StackingRegressor(
        estimators=[
            ('rf', RandomForestRegressor(n_estimators=100, max_depth=8,
                                         random_state=SEED, n_jobs=-1)),
            ('svr', make_pipeline(StandardScaler(), SVR(kernel='rbf', C=10, epsilon=0.2)))
        ],
        final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1)
}

# TODO: adjust chosen hyperparameters based on VALIDATION, not TEST.
# TODO: fit on X_dev/y_dev, then evaluate all models on X_test/y_test ONCE.
# final_rows = []
# for name, model in final_models.items():
#     model.fit(X_dev, y_dev)
#     ...
# display(pd.DataFrame(final_rows).sort_values('Test RMSE'))

# TODO: plot y_test vs prediction and residual vs prediction for the best model.


### Student responses — Part 6

**(a) Final model comparison:** _Insert final table._  
**(b) Diagnostics:** _Insert both plots and interpret patterns._  
**(c) Transfer to crop/field data:** _Propose a target, predictors, evaluation strategy, and limitation._


## Submission checklist

- [ ] All required cells execute in order.
- [ ] Hyperparameter comparison tables and plots are labeled.
- [ ] All interpretation questions have written answers.
- [ ] Final held-out test comparison is completed once after selection.
- [ ] Agroecological application and spatial/temporal limitation discussed.

### Rubric (100 points)
| Part | Points |
|---|---:|
| 1. Data and baseline | 10 |
| 2. Random Forest experiments | 20 |
| 3. Boosting experiment | 20 |
| 4. Early stopping | 15 |
| 5. Stacking | 20 |
| 6. Evaluation and interpretation | 15 |

**Reference:** [Original teaching notebook](https://github.com/acesillinois/cpsc-ese-practicum/blob/main/ENSEMBLMethods.ipynb); [UCI Abalone dataset](https://archive.ics.uci.edu/dataset/1/abalone).
